# Netflix Data Analysis - Visualizations

This notebook creates publication-ready visualizations for the Netflix SQL Analysis project.
Run the SQL queries first and export results to CSV, then use this notebook to generate charts.

In [ ]:
# Setup and imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Style configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Color palette
NETFLIX_RED = '#E50914'
NETFLIX_BLACK = '#141414'
NETFLIX_DARK_GRAY = '#181818'
NETFLIX_GRAY = '#808080'
NETFLIX_LIGHT_GRAY = '#B3B3B3'
NETFLIX_WHITE = '#FFFFFF'

CATEGORICAL_COLORS = [
    '#E50914', '#221F1F', '#FFFFFF', '#B3B3B3', '#808080',
    '#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8',
    '#F7DC6F', '#BB8FCE', '#85C1E9', '#F8B500', '#00CED1'
]

## Load Data

Export your SQL query results to CSV files in the `output/` directory, then update the paths below.

In [ ]:
# Load data from CSV exports (update paths as needed)
# genre_data = pd.read_csv('../output/genre_analysis.csv')
# temporal_data = pd.read_csv('../output/temporal_analysis.csv')
# geographic_data = pd.read_csv('../output/geographic_analysis.csv')
# rating_data = pd.read_csv('../output/rating_analysis.csv')
# duration_data = pd.read_csv('../output/duration_analysis.csv')
# creator_data = pd.read_csv('../output/creator_analysis.csv')

# For demo purposes, create sample data structures
print("Update the file paths above to point to your exported CSV files")
print("Run the SQL queries first, then export results using scripts/export_results.py")

## 1. Genre Distribution Treemap

In [ ]:
def plot_genre_treemap(genre_df):
    """Create a treemap of genre popularity."""
    import squarify
    
    fig, ax = plt.subplots(figsize=(16, 10))
    
    # Top 20 genres
    top_genres = genre_df.head(20)
    sizes = top_genres['total_titles'].values
    labels = [f"{row['genre']}\n{row['total_titles']:,} ({row['pct_of_catalog']:.1f}%)" 
              for _, row in top_genres.iterrows()]
    colors = CATEGORICAL_COLORS[:len(top_genres)]
    
    squarify.plot(sizes=sizes, label=labels, color=colors, alpha=0.8, 
                  text_kwargs={'fontsize': 10, 'fontweight': 'bold'})
    
    ax.set_title('Netflix Genre Popularity - Top 20 Genres', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('../output/genre_treemap.png', dpi=300, bbox_inches='tight', 
                facecolor='white')
    plt.show()

# plot_genre_treemap(genre_data)

## 2. Release Timeline Area Chart

In [ ]:
def plot_release_timeline(temporal_df):
    """Create area chart of content releases over time."""
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Filter recent years
    df = temporal_df[temporal_df['release_year'] >= 1990].copy()
    
    ax.fill_between(df['release_year'], df['movies'], 
                    alpha=0.6, color=NETFLIX_RED, label='Movies')
    ax.fill_between(df['release_year'], df['movies'] + df['tv_shows'], 
                    df['movies'], alpha=0.6, color='#221F1F', label='TV Shows')
    
    ax.plot(df['release_year'], df['movies'] + df['tv_shows'], 
            color=NETFLIX_BLACK, linewidth=2, label='Total')
    
    ax.set_title('Netflix Content Releases Over Time (1990-Present)', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Release Year', fontsize=12)
    ax.set_ylabel('Number of Titles Released', fontsize=12)
    ax.legend(loc='upper left', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Annotate peak
    peak_year = df.loc[df['movies'] + df['tv_shows']).idxmax(), 'release_year']
    peak_val = df['movies'] + df['tv_shows']).max()
    ax.annotate(f'Peak: {peak_val:,} titles ({int(peak_year)})', 
                xy=(peak_year, peak_val), xytext=(peak_year-5, peak_val*1.1),
                arrowprops=dict(arrowstyle='->', color=NETFLIX_RED),
                fontsize=11, fontweight='bold', color=NETFLIX_RED)
    
    plt.tight_layout()
    plt.savefig('../output/release_timeline.png', dpi=300, bbox_inches='tight', 
                facecolor='white')
    plt.show()

## 3. Geographic Content Heatmap

In [ ]:
def plot_geographic_heatmap(geo_df):
    """Create a world heatmap of content by country."""
    
    import plotly.express as px
    
    # Top 50 countries
    top_countries = geo_df.head(50)
    
    fig = px.choropleth(
        top_countries,
        locations='country',
        locationmode='country names',
        color='total_titles',
        hover_name='country',
        hover_data={'movies': True, 'tv_shows': True, 'total_titles': True},
        color_continuous_scale='Reds',
        title='Netflix Content Distribution by Country (Top 50)',
        labels={'total_titles': 'Total Titles'}
    )
    
    fig.update_layout(
        title_font_size=16,
        title_font_color=NETFLIX_BLACK,
        geo=dict(showframe=False, showcoastlines=True, coastlinecolor=NETFLIX_GRAY),
        width=1200,
        height=700
    )
    
    fig.write_html('../output/geographic_heatmap.html')
    fig.show()

## 4. Rating Distribution Donut Chart

In [ ]:
def plot_rating_donut(rating_df):
    """Create donut chart of rating distribution."""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Overall rating distribution
    overall = rating_df.groupby('rating')['count'].sum().sort_values(ascending=False)
    colors_overall = CATEGORICAL_COLORS[:len(overall)]
    
    wedges1, texts1, autotexts1 = ax1.pie(
        overall.values,
        labels=overall.index,
        autopct='%1.1f%%',
        colors=colors_overall,
        startangle=90,
        pctdistance=0.85,
        wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2)
    )
    ax1.set_title('Overall Rating Distribution', fontsize=14, fontweight='bold', pad=20)
    
    # By type
    movie_ratings = rating_df[rating_df['type'] == 'Movie'].set_index('rating')['count']
    show_ratings = rating_df[rating_df['type'] == 'TV Show'].set_index('rating')['count']
    
    x = np.arange(len(movie_ratings.index))
    width = 0.35
    
    ax2.bar(x - width/2, movie_ratings.values, width, label='Movies', 
            color=NETFLIX_RED, alpha=0.8)
    ax2.bar(x + width/2, show_ratings.values, width, label='TV Shows', 
            color=NETFLIX_BLACK, alpha=0.8)
    
    ax2.set_xlabel('Rating')
    ax2.set_ylabel('Count')
    ax2.set_title('Rating Distribution by Content Type', fontsize=14, fontweight='bold', pad=20)
    ax2.set_xticks(x)
    ax2.set_xticklabels(movie_ratings.index, rotation=45, ha='right')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('../output/rating_distribution.png', dpi=300, bbox_inches='tight', 
                facecolor='white')
    plt.show()

## 5. Duration Analysis - Box Plots

In [ ]:
def plot_duration_boxplots(duration_df):
    """Create box plots for movie duration and TV show seasons."""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    
    # Movie duration boxplot by genre (top 10)
    movie_data = duration_df[duration_df['type'] == 'Movie']
    if len(movie_data) > 0:
        top_genres_movie = movie_data.groupby('genre')['duration'].median().nlargest(10).index
        plot_data_m = [movie_data[movie_data['genre'] == g]['duration'].values for g in top_genres_movie]
        
        bp1 = ax1.boxplot(plot_data_m, labels=top_genres_movie, patch_artist=True,
                          widths=0.6, showfliers=False)
        for patch, color in zip(bp1['boxes'], CATEGORICAL_COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax1.set_title('Movie Duration by Genre (Top 10)', fontsize=13, fontweight='bold')
        ax1.set_ylabel('Duration (minutes)')
        ax1.tick_params(axis='x', rotation=45)
        ax1.grid(True, alpha=0.3, axis='y')
    
    # TV Show seasons boxplot by genre
    tv_data = duration_df[duration_df['type'] == 'TV Show']
    if len(tv_data) > 0:
        top_genres_tv = tv_data.groupby('genre')['duration'].median().nlargest(10).index
        plot_data_t = [tv_data[tv_data['genre'] == g]['duration'].values for g in top_genres_tv]
        
        bp2 = ax2.boxplot(plot_data_t, labels=top_genres_tv, patch_artist=True,
                          widths=0.6, showfliers=False)
        for patch, color in zip(bp2['boxes'], CATEGORICAL_COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax2.set_title('TV Show Seasons by Genre (Top 10)', fontsize=13, fontweight='bold')
        ax2.set_ylabel('Number of Seasons')
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('../output/duration_boxplots.png', dpi=300, bbox_inches='tight', 
                facecolor='white')
    plt.show()

## 6. Top Directors Bar Chart

In [ ]:
def plot_top_directors(creator_df):
    """Create horizontal bar chart of top directors."""
    
    director_data = creator_df[creator_df['role'] == 'director'].head(20)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    y_pos = np.arange(len(director_data))
    colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(director_data)))
    
    bars = ax.barh(y_pos, director_data['total_titles'], color=colors, 
                   edgecolor=NETFLIX_BLACK, linewidth=0.5, alpha=0.9)
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, director_data['total_titles'])):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, 
                f'{int(val)}', va='center', fontweight='bold', fontsize=10)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(director_data['name'])
    ax.invert_yaxis()
    ax.set_xlabel('Number of Titles on Netflix')
    ax.set_title('Top 20 Directors on Netflix', fontsize=16, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig('../output/top_directors.png', dpi=300, bbox_inches='tight', 
                facecolor='white')
    plt.show()

## 7. Genre Trends Over Time (Interactive)

In [ ]:
def plot_genre_trends_interactive(genre_trend_df):
    """Create interactive line chart of genre trends over time."""
    
    import plotly.graph_objects as go
    
    fig = go.Figure()
    
    top_genres = genre_trend_df.groupby('genre')['count'].sum().nlargest(10).index
    
    for i, genre in enumerate(top_genres):
        genre_data = genre_trend_df[genre_trend_df['genre'] == genre]
        fig.add_trace(go.Scatter(
            x=genre_data['release_year'],
            y=genre_data['count'],
            mode='lines+markers',
            name=genre,
            line=dict(width=3, color=CATEGORICAL_COLORS[i % len(CATEGORICAL_COLORS)]),
            marker=dict(size=6)
        ))
    
    fig.update_layout(
        title='Top 10 Genre Trends Over Time',
        title_font_size=16,
        title_font_color=NETFLIX_BLACK,
        xaxis_title='Release Year',
        yaxis_title='Number of Titles',
        hovermode='x unified',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.02
        ),
        width=1100,
        height=600,
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='white'
    )
    
    fig.write_html('../output/genre_trends.html')
    fig.show()

## 8. Content Type & Rating Matrix Heatmap

In [ ]:
def plot_type_rating_heatmap(rating_df):
    """Create heatmap of content type vs rating."""
    
    pivot = rating_df.pivot_table(
        index='rating', 
        columns='type', 
        values='count', 
        fill_value=0
    )
    
    # Normalize to percentages within each type
    pivot_pct = pivot.div(pivot.sum(axis=0), axis=1) * 100
    
    fig, ax = plt.subplots(figsize=(10, 10))
    
    sns.heatmap(pivot_pct, 
                annot=True, 
                fmt='.1f',
                cmap='Reds',
                cbar_kws={'label': 'Percentage within Type'},
                linewidths=0.5,
                linecolor='white',
                ax=ax)
    
    ax.set_title('Rating Distribution by Content Type (%)', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('')
    ax.set_ylabel('')
    
    plt.tight_layout()
    plt.savefig('../output/type_rating_heatmap.png', dpi=300, bbox_inches='tight', 
                facecolor='white')
    plt.show()

## 9. Monthly Addition Seasonality

In [ ]:
def plot_monthly_seasonality(monthly_df):
    """Create polar/radar chart for monthly addition patterns."""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), 
                                    subplot_kw=dict(projection='polar'))
    
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    angles = np.linspace(0, 2*np.pi, 12, endpoint=False)
    
    # Movies
    movie_monthly = monthly_df[monthly_df['type'] == 'Movie'].sort_values('month')
    values_m = movie_monthly['count'].values
    values_m = np.append(values_m, values_m[0])
    angles_closed = np.append(angles, angles[0])
    
    ax1.plot(angles_closed, values_m, 'o-', linewidth=2, color=NETFLIX_RED)
    ax1.fill(angles_closed, values_m, alpha=0.25, color=NETFLIX_RED)
    ax1.set_xticks(angles)
    ax1.set_xticklabels(months)
    ax1.set_title('Movies Added by Month', fontsize=13, fontweight='bold', pad=20)
    
    # TV Shows
    tv_monthly = monthly_df[monthly_df['type'] == 'TV Show'].sort_values('month')
    values_t = tv_monthly['count'].values
    values_t = np.append(values_t, values_t[0])
    
    ax2.plot(angles_closed, values_t, 'o-', linewidth=2, color=NETFLIX_BLACK)
    ax2.fill(angles_closed, values_t, alpha=0.25, color=NETFLIX_BLACK)
    ax2.set_xticks(angles)
    ax2.set_xticklabels(months)
    ax2.set_title('TV Shows Added by Month', fontsize=13, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.savefig('../output/monthly_seasonality.png', dpi=300, bbox_inches='tight', 
                facecolor='white')
    plt.show()

## 10. Executive Dashboard Summary

In [ ]:
def create_executive_dashboard():
    """Create a comprehensive executive dashboard."""
    
    fig = plt.figure(figsize=(20, 14))
    gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)
    
    # Color scheme
    bg_color = '#F5F5F5'
    fig.patch.set_facecolor(bg_color)
    
    # 1. Key Metrics (Top row - 4 KPI cards)
    kpi_data = [
        ('82,000+', 'Total Titles', NETFLIX_RED),
        ('68%', 'Movies', NETFLIX_BLACK),
        ('32%', 'TV Shows', '#4ECDC4'),
        ('23%', 'Top Genre: Drama', '#FFA07A')
    ]
    
    for i, (value, label, color) in enumerate(kpi_data):
        ax = fig.add_subplot(gs[0, i])
        ax.text(0.5, 0.6, value, ha='center', va='center', 
                fontsize=32, fontweight='bold', color=color,
                transform=ax.transAxes)
        ax.text(0.5, 0.2, label, ha='center', va='center', 
                fontsize=12, color=NETFLIX_GRAY,
                transform=ax.transAxes)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
        # Add border
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(color)
            spine.set_linewidth(2)
    
    # 2. Genre Distribution (Row 2, Left - spans 2)
    ax2 = fig.add_subplot(gs[1, :2])
    # Placeholder for genre bar chart
    ax2.text(0.5, 0.5, 'Genre Distribution Bar Chart', 
             ha='center', va='center', fontsize=14, color=NETFLIX_GRAY)
    ax2.set_title('Top 10 Genres', fontsize=13, fontweight='bold')
    ax2.axis('off')
    
    # 3. Release Timeline (Row 2, Right - spans 2)
    ax3 = fig.add_subplot(gs[1, 2:])
    ax3.text(0.5, 0.5, 'Release Timeline Area Chart', 
             ha='center', va='center', fontsize=14, color=NETFLIX_GRAY)
    ax3.set_title('Content Releases Over Time', fontsize=13, fontweight='bold')
    ax3.axis('off')
    
    # 4. Rating Distribution (Row 3, Left)
    ax4 = fig.add_subplot(gs[2, 0])
    ax4.text(0.5, 0.5, 'Rating Donut Chart', 
             ha='center', va='center', fontsize=14, color=NETFLIX_GRAY)
    ax4.set_title('Rating Distribution', fontsize=12, fontweight='bold')
    ax4.axis('off')
    
    # 5. Top Countries (Row 3, Middle-Left)
    ax5 = fig.add_subplot(gs[2, 1])
    ax5.text(0.5, 0.5, 'Top Countries Bar Chart', 
             ha='center', va='center', fontsize=14, color=NETFLIX_GRAY)
    ax5.set_title('Top 10 Countries', fontsize=12, fontweight='bold')
    ax5.axis('off')
    
    # 6. Duration Stats (Row 3, Middle-Right)
    ax6 = fig.add_subplot(gs[2, 2])
    ax6.text(0.5, 0.5, 'Duration Box Plots', 
             ha='center', va='center', fontsize=14, color=NETFLIX_GRAY)
    ax6.set_title('Duration Analysis', fontsize=12, fontweight='bold')
    ax6.axis('off')
    
    # 7. Key Insight (Row 3, Right)
    ax7 = fig.add_subplot(gs[2, 3])
    insights = [
        "• Drama dominates at 23% of catalog",
        "• 2019-2021: Peak content acquisition",
        "• US produces 35% of content",
        "• TV-MA is top rating (31%)",
        "• Avg movie: 98 min, Avg show: 2.3 seasons"
    ]
    ax7.text(0.05, 0.9, 'KEY INSIGHTS', ha='left', va='top', 
             fontsize=13, fontweight='bold', color=NETFLIX_RED,
             transform=ax7.transAxes)
    for i, insight in enumerate(insights):
        ax7.text(0.05, 0.75 - i*0.12, insight, ha='left', va='top', 
                 fontsize=11, color=NETFLIX_BLACK,
                 transform=ax7.transAxes)
    ax7.axis('off')
    
    plt.suptitle('NETFLIX CONTENT ANALYSIS - EXECUTIVE DASHBOARD', 
               fontsize=20, fontweight='bold', color=NETFLIX_BLACK, y=0.98)
    
    plt.savefig('../output/executive_dashboard.png', dpi=300, bbox_inches='tight', 
                facecolor=bg_color)
    plt.show()

# create_executive_dashboard()

## Export All Visualizations

In [ ]:
def export_all_visualizations():
    """Run all visualization functions (update with actual data)."""
    print("To generate visualizations:")
    print("1. Run SQL queries in order (01-10)")
    print("2. Export results to CSV using scripts/export_results.py")
    print("3. Load CSVs in this notebook (update paths in cell 2)")
    print("4. Call each plot function with your dataframes")
    print("5. Charts save to ../output/ as PNG and HTML")

# export_all_visualizations()